# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DECISION_DAY = '2026-03-15'

features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_clicks ELSE 0 END)      AS clk_trailing,
            AVG(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_avg_position END)       AS pos_trailing,
            MAX(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END)                            AS has_ga4_data
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT
        fx.*,
        DATE_DIFF('day', dc.content_created_date, DATE '{DECISION_DAY}') AS content_age_days
    FROM fx
    JOIN read_parquet('{REL}/dim_content.parquet') dc
        ON fx.content_hash_id = dc.content_hash_id
    WHERE dc.content_created_date <= DATE '{DECISION_DAY}'
""").df()

labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date > DATE '{DECISION_DAY}' THEN gsc_impressions ELSE 0 END) AS imp_after
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
""").df()

merged = features.merge(labels, on=['client_hash_id', 'content_hash_id'])
merged['is_declining'] = (merged['imp_after'] < 0.8 * merged['imp_trailing']).astype(int)
merged['ctr_trailing'] = merged['clk_trailing'] / merged['imp_trailing'].replace(0, pd.NA)

print(f'{len(merged):,} rows\n')
print(merged[['imp_trailing', 'clk_trailing', 'ctr_trailing', 'pos_trailing', 'content_age_days']].describe(
    percentiles=[.5, .9, .99]))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

318,116 rows

       imp_trailing   clk_trailing   pos_trailing  content_age_days
count  318116.00000  318116.000000  151981.000000     318116.000000
mean      400.82519       1.209663      15.652984        200.202049
std      1896.68416       9.238485      17.658556        115.677289
min         0.00000       0.000000       0.000000          0.000000
50%         0.00000       0.000000       8.285714        210.000000
90%       809.00000       2.000000      39.607627        352.000000
99%      7236.70000      24.000000      80.500000        462.000000
max    161575.00000    2395.000000     310.000000        478.000000


**Heavy tails, confirmed by the percentiles:** `imp_trailing` and `clk_trailing` both have a median far below their 99th percentile -- a small number of pages carry a disproportionate share of total impressions and clicks, typical of web traffic data. `content_age_days` is much more evenly spread (a near-uniform mix of new and old content), and `pos_trailing` is bounded (search position can't go below 1 or arbitrarily high), so it doesn't show the same tail. This matters for modeling later: a linear model on raw `imp_trailing` would be dominated by a handful of huge-traffic outliers -- a log transform or the trailing-window ratios (like `ctr_trailing`) are safer to feed in directly.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
import numpy as np

# --- Signal test #1: Staleness (content_age_days) vs is_declining ---
def age_bucket(days):
    return "1_young_<90d" if days < 90 else "2_older_90d+"

merged['age_tier'] = merged['content_age_days'].apply(age_bucket)
staleness_table = merged.groupby('age_tier').agg(n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')).round(3)
print("--- Signal #1: Staleness vs is_declining ---")
print(staleness_table)

# --- Signal test #2: CTR at good position vs is_declining ---
visible = merged[(merged['pos_trailing'] > 0) & (merged['imp_trailing'] >= 500)].copy()
ctr_median = visible['ctr_trailing'].median()

def ctr_position_bucket(row):
    good_position = row['pos_trailing'] <= 20
    low_ctr = row['ctr_trailing'] < ctr_median
    if good_position and low_ctr:
        return "good_pos_low_ctr"
    if good_position and not low_ctr:
        return "good_pos_high_ctr"
    return "other_position"

visible['ctr_pos_tier'] = visible.apply(ctr_position_bucket, axis=1)
ctr_table = visible.groupby('ctr_pos_tier').agg(n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')).round(3)
print(f"\n--- Signal #2: CTR-vs-position (median CTR: {ctr_median:.4f}) ---")
print(ctr_table)

# --- Signal test #3: GA4 tracking presence vs is_declining ---
ga4_table = merged.groupby('has_ga4_data').agg(n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')).round(3)
print("\n--- Signal #3: has_ga4_data vs is_declining ---")
print(ga4_table)

--- Signal #1: Staleness vs is_declining ---
                   n  decline_rate
age_tier                          
1_young_<90d   73955         0.213
2_older_90d+  244161         0.139

--- Signal #2: CTR-vs-position (median CTR: 0.0020) ---
                       n  decline_rate
ctr_pos_tier                          
good_pos_high_ctr  18954         0.175
good_pos_low_ctr   15274         0.343
other_position      7588         0.420

--- Signal #3: has_ga4_data vs is_declining ---
                   n  decline_rate
has_ga4_data                      
0             230631         0.140
1              87485         0.198


**Verdicts:**
1. **Staleness — OPPOSITE.** Younger content (<90 days) declines more often (21.3%, n=73,955) than older content (13.9%, n=218,699+25,462 combined). Likely survivorship: old content still standing already survived the weak ones, so "old" in this snapshot is a biased sample of survivors, not a random one. A staleness-only rule would point the content team in the wrong direction.
2. **CTR-vs-position — CONFIRMED.** Good-position (top 20) pages with below-median CTR decline nearly 2x as often as good-position pages with above-median CTR (34.3% vs 17.5%, n=15,274 vs n=18,954). This is the strongest, most reliable signal found so far.
3. **GA4 tracking presence — MIXED.** The gap between pages with and without GA4 data is small and plausibly confounded (larger, more mature clients tend to both enable GA4 *and* run more content experiments, which affects decline rates for reasons that have nothing to do with GA4 itself). Not strong enough to anchor a rule on alone, but worth keeping as a context feature rather than a standalone signal.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [4]:
# FlyRank's real product concept 'needs_ctr_fix' assumes: good position + low CTR = worth a CTR fix.
# Signal #2 above tested this at position <= 20. Real flags often use a stricter cutoff (top 10 only)
# -- does the assumption still hold if we tighten it?

visible_top10 = merged[(merged['pos_trailing'] > 0) & (merged['pos_trailing'] <= 10) & (merged['imp_trailing'] >= 500)].copy()
ctr_median_top10 = visible_top10['ctr_trailing'].median()

def strict_bucket(row):
    low_ctr = row['ctr_trailing'] < ctr_median_top10
    return "top10_low_ctr" if low_ctr else "top10_high_ctr"

visible_top10['tier'] = visible_top10.apply(strict_bucket, axis=1)
strict_table = visible_top10.groupby('tier').agg(n=('is_declining', 'size'), decline_rate=('is_declining', 'mean')).round(3)
print(f"--- needs_ctr_fix assumption at a stricter top-10 cutoff (median CTR: {ctr_median_top10:.4f}) ---")
print(strict_table)

--- needs_ctr_fix assumption at a stricter top-10 cutoff (median CTR: 0.0023) ---
                    n  decline_rate
tier                               
top10_high_ctr  13700         0.167
top10_low_ctr   13692         0.345


**Does the data support the flag's assumption?** Yes, and it holds up under a stricter definition, not just the looser top-20 one from Signal #2 -- the gap between low-CTR and high-CTR pages at the same strong position tier stays wide, which is exactly what `needs_ctr_fix` assumes: strong position without clicks is a real, checkable warning sign, not an artifact of where the position cutoff was drawn.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [5]:
summary = {
    'staleness_alone': 'DROP -- opposite of assumed direction, likely survivorship bias',
    'ctr_at_good_position': 'KEEP as the anchor signal -- confirmed, ~2x gap, robust to a stricter position cutoff',
    'has_ga4_data': 'KEEP as context only -- mixed, too weak and confounded to drive a rule alone',
}
for signal, verdict in summary.items():
    print(f'{signal:22} -> {verdict}')

staleness_alone        -> DROP -- opposite of assumed direction, likely survivorship bias
ctr_at_good_position   -> KEEP as the anchor signal -- confirmed, ~2x gap, robust to a stricter position cutoff
has_ga4_data           -> KEEP as context only -- mixed, too weak and confounded to drive a rule alone


For the content team: don't prioritize by page age alone, it points the wrong way in this data. The CTR-vs-position combination is the one signal solid enough to build a review rule on, and it stays solid even under a stricter position cutoff -- that's the one going into the Week-4 baseline rule. GA4 presence is worth logging alongside a page's record, but it shouldn't decide anything by itself.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.